<a href="https://colab.research.google.com/github/Jess-MY998/ADALL_github/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai pandas numpy matplotlib seaborn python-dotenv

In [4]:
import pandas as pd
import numpy as np


In [5]:
github_raw_url = 'https://raw.githubusercontent.com/Jess-MY998/ADALL_github/refs/heads/main/laptop_prices_2024_sgd_TL.csv'

In [6]:
try:
  df = pd.read_csv(github_raw_url)
  print("Successfully loaded data from Github!")
  display(df.head())
except Exception as e:
  print(f"Error loading data from Github: {e}")
  print("Please ensure the URL is correct and the file format compatible with 'pd.read_csv'.")

Successfully loaded data from Github!


,Brand,Model,CPU,GPU,RAM_GB,Storage_Type,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch,Discount_percent,Price_SGD,Brand_Discount,Member_Discount
0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,SSD,256,False,1.56,16.0,3.27,2413.36,5,144.80
1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,SSD,1024,True,1.45,14.0,5.03,1773.75,5,124.16
2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,SSD,2048,False,1.34,14.0,4.41,1634.07,5,98.04
3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,SSD,4096,True,1.18,13.3,2.16,2362.59,5,118.13
4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,SSD,1024,True,1.31,14.0,6.93,2218.55,5,155.30


In [7]:
from google.colab import userdata
from openai import OpenAI

# Load key from Google Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=api_key,
)

In [8]:
# Convert the first few rows to a string to send to Open AI.
data_preview = df.head().to_string()
print(data_preview)

  Brand                Model                        CPU              GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
0  Acer             Aspire 5           Intel i9-14900HK  NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              3.27    2413.36               5           144.80
1  Acer              Nitro 5         AMD Ryzen 9 8900HX  AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              5.03    1773.75               5           124.16
2  Acer              Nitro 5          AMD Ryzen 5 8600H  NVIDIA RTX 4050      32          SSD        2048        False       1.34              14.0              4.41    1634.07               5            98.04
3  Acer        TravelMate P6  Intel Core Ultra 7 15500H  NVIDIA RTX 4060      16          SSD        4096         True       1.18              13.3             

In [12]:
response = client.responses.create(
  model="gpt-5-mini",
  instructions="You are an expert data scientist with extensive knowledge of predictive analysis and linear regression.",
  input=f"Dataset: Laptop Price Prdiction (1000 samples, 14 features, target is Price_SGD)\nHere are the first 10 rows of the dataset:\n{data_preview}]",
)
print(response.output_text)

Thanks — this looks like a good dataset to predict Price_SGD. Before I proceed, quick question: do you want a full working model and code (train + CV + final predictions) or high-level guidance and a recommended pipeline? I can do either. Below I outline a practical end-to-end approach plus a compact scikit-learn/XGBoost pipeline you can run.

Recommended approach (step-by-step)
1. Exploratory data analysis
   - Check target distribution (Price_SGD). If right-skewed, consider log(1+x) transform.
   - Compute correlations and pairwise plots (numerical) and median price by categorical levels (Brand, CPU, GPU, Model).
   - Look for duplicates, missing values, outliers and data leakage (features derived from price).
   - Check cardinality: Model/CPU/GPU likely high-cardinality.

2. Feature engineering
   - Numeric features: RAM_GB, Storage_GB, Weight_kg, Screen_Size_inch, Discount_percent, Brand_Discount, Member_Discount.
   - Binary: Touchscreen -> 0/1.
   - Categorical low-cardinality: B